## Environment Set Up

In [ ]:
from langchain_openai import OpenAIEmbeddings
# from langchain.chains import GraphCypherQAChain
from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores.neo4j_vector import Neo4jVector

In [ ]:
# check if GPU is available
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
# environment set up
from dotenv import load_dotenv
load_dotenv()

## Connect with the Neo4j Graph Database

In [ ]:
# Connect with the neo4j database

import os

# Neo4j connection details
url = os.getenv("NEO4J_URI")
username = os.getenv("NEO4J_USERNAME")
password = os.getenv("NEO4J_PASSWORD")

# specifying the URL to the Neo4j database server, and the credentials needed to access it
graph = Neo4jGraph(
    url= url,
    username= username,
    password= password
)

In [ ]:
print(graph.schema)

In [ ]:

# # create a connection to Neo4j -- neo4j style
# from neo4j import GraphDatabase

# driver = GraphDatabase.driver(
#     url,
#     auth=(username, password)
# )
# driver.verify_connectivity()

In [ ]:
# # test the connection by querying the database

# results = graph.query("""
# MATCH (n:owl__Class)
# where n.rdfs__label = "DataController" or n.rdfs__label = "DataProcessor"
# Return n.rdfs__label, n.rdfs__comment
# """)
# print(results)

## Embedding Model

In [ ]:
# get the open_ai_key
OPENAI_API_KEY = os.environ['OPENAI_API_KEY']

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_neo4j import GraphCypherQAChain

# create open-ai llm instance
llm_4omini = ChatOpenAI(api_key = OPENAI_API_KEY, model = "gpt-4o-mini", temperature = 0)

In [ ]:
from langchain_openai import OpenAIEmbeddings

embedder = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

Neo4j style --

In [ ]:
# from langchain_openai import OpenAIEmbeddings

# class OpenAIEmbeddingAdapter(OpenAIEmbeddings):

#     def embed_documents(self, texts):
#         return super().embed_documents(texts)

# embedder = OpenAIEmbeddingAdapter(
#     model="text-embedding-3-small"
# )

## Vector Indexing

create neo4j vector store (vector index) 

In [ ]:
vector_store = Neo4jVector.from_existing_graph(
    embedding=embedder,
    url= url,
    username= username,
    password=password,
    index_name="resource_label_index",
    node_label="Resource",
    text_node_properties=["rdfs__label"],
    embedding_node_property="labelEmbedding",
)

~ Check the embedding is in the neo4j database (run the query in the neo4j database)

MATCH (n:Resource)  
WHERE n.labelEmbedding IS NOT NULL  
RETURN n.rdfs__label, n.labelEmbedding[0..5]  
LIMIT 5  

## Test Similarity Search

In [ ]:
results = vector_store.similarity_search(
    "What is biometric data under PDPA?",
    k=5
)

for r in results:
    print(r.page_content)
    print(r.metadata)